In [ ]:
# gds.ipynb — экспорт GDS для фабрикации
#
# Этот ноутбук:
# - НЕ обучает и НЕ оптимизирует
# - берёт `pos` из preset (outputs/<run>/results/solutions/<preset>.json)
# - экспортирует GDS + txt-описание в outputs/<run>/gds/

%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

from pcm_pix.run import start_run
from pcm_pix.optimize import load_solution
from pcm_pix.gds import GDSFabCfg, export_fabrication_gds

# --- RUN SELECTION ---
RUN_NAME = "PCM_bagel_2025"   # outputs/<run_name>
DATA_DIR = "data"

run = start_run(outputs_dir="outputs", run_name=RUN_NAME)
CFG: dict = {"run_name": RUN_NAME, "data_dir": DATA_DIR}

cfg_path = run.results / "config.json"
if cfg_path.exists():
    CFG.update(json.loads(cfg_path.read_text(encoding="utf-8")))

CFG["data_dir"] = DATA_DIR
CFG

In [ ]:
# --- Load pos from preset ---
preset_name = CFG.get("preset_name", "pos_server_2026_03_04_night")

sol_path = run.results / "solutions" / f"{preset_name}.json"
if not sol_path.exists():
    raise FileNotFoundError(f"Preset not found: {sol_path}")

pos, cost, meta = load_solution(sol_path)

Nn = int(CFG.get("Nn", 11))
if len(pos) < 3 * Nn:
    raise ValueError(f"pos too short: len(pos)={len(pos)} for Nn={Nn}")

a_m = pos[0:Nn] * 1e-9
d_m = pos[Nn:2*Nn] * 1e-9
b_m = pos[2*Nn:3*Nn] * 1e-9

# как в оптимизации: b < 50nm -> 0
b_min = float(CFG.get("b_min_m", 50e-9))
b_m = b_m.copy()
b_m[b_m < b_min] = 0.0

a_m, d_m, b_m, float(cost) if cost is not None else None

In [ ]:
# --- Export GDS (fabrication) ---
# Параметры как в 3rd art:
# l = 20 µm, L = 11*l
cfg_gds = GDSFabCfg(
    name=str(CFG.get("gds_name", f"gds_{preset_name}")),
    l_m=float(CFG.get("gds_l_m", 20e-6)),
    L_m=float(CFG.get("gds_L_m", 11 * 20e-6)),
    layer_mode=str(CFG.get("gds_layer_mode", "fab")),
)

meta_out = {
    "preset_name": preset_name,
    "cost": cost,
    "source_meta": meta,
}

# Пишем в outputs/<run>/gds/
gds_path, txt_path = export_fabrication_gds(
    a_m=a_m,
    d_m=d_m,
    b_m=b_m,
    out_dir=run.gds,
    cfg=cfg_gds,
    meta=meta_out,
)

gds_path, txt_path